# TPG366 Live Pressure Monitor

Live monitoring of all 6 TPG366 pressure sensors with web-based plotting (opens in Chrome tab).

**Sensors:** CH1=IF1, CH2=IF2, CH3=Q1, CH4=QMS, CH5=Transfer, CH6=Depo

**Features:**
- 6 individual subplots with shared time axis, independent y-axes (log scale)
- ~1 Hz update rate
- All values logged to file
- Web-based plot opens in browser tab via Bokeh server

## 1. Imports

In [1]:
import sys
import os
import logging
import threading
import webbrowser
from datetime import datetime, timedelta
from pathlib import Path

from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, DatetimeTickFormatter, Range1d, Label
from bokeh.server.server import Server
from tornado.ioloop import IOLoop

sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.pfeiffer.tpg366.tpg366 import TPG366

print("Imports done.")

Imports done.


## 2. Configuration

In [2]:
# TPG366 connection
COM_PORT = "COM25"
DEVICE_ADDRESS = 10

# Monitoring
POLL_INTERVAL_MS = 1000  # 1 Hz update rate
ROLLING_WINDOW_MIN = 1  # minutes of data visible in plot
MAX_POINTS = 3600        # max data points kept in memory (~1 hour at 1 Hz)

# Bokeh server
BOKEH_PORT = 5006

# Sensor labels
SENSOR_LABELS = {
    1: "IF1",
    2: "IF2",
    3: "Q1",
    4: "QMS",
    5: "Transfer",
    6: "Depo",
}

print("Configuration set.")

Configuration set.


## 3. Setup Logger

In [3]:
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"020_tpg366_monitor_{timestamp}.log"

logger = logging.getLogger(f"TPG366_Monitor_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

2026-04-17 09:32:19,619 - INFO - Logger initialized.


Log file: C:\Users\ESIBDlab\PycharmProjects\esibd_bs\debugging\logs\020_tpg366_monitor_20260417_093219.log


## 4. Connect TPG366

In [4]:
gauge = TPG366(
    device_id="tpg366",
    port=COM_PORT,
    device_address=DEVICE_ADDRESS,
    baudrate=115200,
    logger=logger,
)
gauge.connect()
print(f"TPG366 connected on {COM_PORT}")

2026-04-17 09:32:21,812 - INFO - Connecting to Pfeiffer device tpg366 on COM25
2026-04-17 09:32:21,839 - INFO - Successfully connected to device at address 10


TPG366 connected on COM25


## 5. Verify Connection

In [5]:
print(f"Serial Number:    {gauge.get_serial_number()}")
print(f"Device Name:      {gauge.get_device_name()}")
print(f"Firmware Version: {gauge.get_firmware_version()}")
print()
pressures = gauge.read_all_pressures()
for ch, val in pressures.items():
    print(f"  CH{ch} ({SENSOR_LABELS[ch]}): {val} hPa")

Serial Number:    44866963
Device Name:      TPG366
Firmware Version: 010400

  CH1 (IF1): 907.4 hPa
  CH2 (IF2): 1004.0 hPa
  CH3 (Q1): 997.7 hPa
  CH4 (QMS): 0.0 hPa
  CH5 (Transfer): 0.0 hPa
  CH6 (Depo): 0.0 hPa


## 6. Launch Live Plot (opens in Chrome)

Running this cell starts a Bokeh server in a background thread and opens the live plot in Chrome.

**Stop monitoring:** Run the next cell (Section 7) to stop the server.

In [6]:
# Suppress console logging during live monitoring (file logging continues)
console_handler.setLevel(logging.WARNING)


def make_document(doc):
    """Build the Bokeh document with 6 pressure subplots."""

    # Shared data sources for each sensor
    sources = {}
    for ch in range(1, 7):
        sources[ch] = ColumnDataSource(data={"time": [], "pressure": []})

    # Explicit x-range so the rolling window works (DataRange1d auto-adjusts and ignores manual bounds)
    now = datetime.now()
    x_range = Range1d(start=now - timedelta(minutes=ROLLING_WINDOW_MIN), end=now)

    # Create 6 figures stacked vertically, sharing x-axis
    figures = []
    labels = {}
    for ch in range(1, 7):
        label = SENSOR_LABELS[ch]
        p = figure(
            title=f"CH{ch}: {label}",
            x_axis_type="datetime",
            y_axis_type="log",
            height=180,
            width=1000,
            x_range=x_range,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        p.line("time", "pressure", source=sources[ch], line_width=2)
        p.yaxis.axis_label = "Pressure [hPa]"
        p.xaxis.formatter = DatetimeTickFormatter(
            seconds="%H:%M:%S",
            minsec="%H:%M:%S",
            minutes="%H:%M:%S",
            hourmin="%H:%M:%S",
            hours="%H:%M:%S",
        )

        # Current value label in top-right corner of each plot
        value_label = Label(
            x=10, y=10,
            x_units="screen", y_units="screen",
            text="---",
            text_font_size="14px",
            text_font_style="bold",
            text_color="#333",
            background_fill_color="white",
            background_fill_alpha=0.7,
        )
        p.add_layout(value_label)
        labels[ch] = value_label

        # Only show x-axis label on bottom plot
        if ch < 6:
            p.xaxis.axis_label = ""
        else:
            p.xaxis.axis_label = "Time"

        figures.append(p)

    layout = column(*figures, sizing_mode="stretch_width")
    doc.add_root(layout)
    doc.title = "TPG366 Pressure Monitor"

    def update():
        """Periodic callback: read pressures, log, and update plot."""
        try:
            pressures = gauge.read_all_pressures()
            now = datetime.now()

            for ch in range(1, 7):
                val = pressures.get(ch)
                if val is not None:
                    # Log using same format as custom_logger
                    logger.info(
                        f"tpg366//{COM_PORT}//Sensor_CH{ch}_Press={val}//hPa"
                    )

                    # Stream new data point (with rollover to limit memory)
                    sources[ch].stream(
                        {"time": [now], "pressure": [val]},
                        rollover=MAX_POINTS,
                    )

                    # Update current value label
                    if val == 0.0:
                        val_str = "0 hPa"
                    elif val < 0.01:
                        val_str = f"{val:.2e} hPa"
                    elif val < 100:
                        val_str = f"{val:.3f} hPa"
                    else:
                        val_str = f"{val:.1f} hPa"
                    labels[ch].text = val_str

            # Update x-range to show rolling window
            x_range.start = now - timedelta(minutes=ROLLING_WINDOW_MIN)
            x_range.end = now

        except Exception as e:
            logger.error(f"Update failed: {e}")

    doc.add_periodic_callback(update, POLL_INTERVAL_MS)


# Run Bokeh server on its own IOLoop in a background thread (avoids conflict with Jupyter's event loop)
def _start_server():
    io_loop = IOLoop()
    io_loop.make_current()
    server = Server(
        {'/': make_document},
        port=BOKEH_PORT,
        io_loop=io_loop,
        allow_websocket_origin=[f"localhost:{BOKEH_PORT}"],
    )
    server.start()
    io_loop.start()

server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

# Open in Chrome specifically
import time
time.sleep(1)  # give server a moment to start
url = f"http://localhost:{BOKEH_PORT}/"
try:
    chrome = webbrowser.get("C:/Program Files/Google/Chrome/Application/chrome.exe %s")
    chrome.open(url)
    print(f"Opened Chrome at {url}")
except webbrowser.Error:
    webbrowser.open(url)
    print(f"Chrome not found, opened default browser at {url}")

print(f"Bokeh server running at {url}")
print("Server runs in background. Run the next cell to stop it.")

C:\Users\ESIBDlab\AppData\Local\Temp\ipykernel_11420\2372328816.py:112: DeprecationWarning: make_current is deprecated; start the event loop first
  io_loop.make_current()


Opened Chrome at http://localhost:5006/
Bokeh server running at http://localhost:5006/
Server runs in background. Run the next cell to stop it.


## 6.5 Manual Control

In [ ]:
gauge._query_channel_parameter(4, 349)

In [7]:
def activate_ch(channel):
    value = gauge.data_converter.int_2_u_short_int(1)
    gauge._set_channel_parameter(channel, 41, value)

def deactivate_ch(channel):
    value = gauge.data_converter.int_2_u_short_int(0)
    gauge._set_channel_parameter(channel, 41, value)

In [8]:
activate_ch(4)

In [9]:
activate_ch(5)

In [12]:
activate_ch(6)

In [13]:
deactivate_ch(4)

In [14]:
deactivate_ch(5)

In [15]:
deactivate_ch(6)

In [ ]:
gauge.read_all_pressures()

## 7. Stop Server

In [ ]:
console_handler.setLevel(logging.DEBUG)
logger.info("Monitoring stopped.")
print("Monitoring stopped. Log file logging complete.")

## 8. Disconnect

In [16]:
gauge.disconnect()
print("TPG366 disconnected.")

2026-04-17 15:21:44,054 - ERROR - Failed to query channel 1 parameter 740: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-04-17 15:21:44,055 - WARNING - Failed to read pressure from channel 1: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-04-17 15:21:44,055 - ERROR - Failed to query channel 2 parameter 740: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-04-17 15:21:44,056 - WARNING - Failed to read pressure from channel 2: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-04-17 15:21:44,056 - ERROR - Failed to query channel 3 parameter 740: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-04-17 15:21:44,057 - WARNING - Failed to read pressure from channel 3: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-04-17 15:21:44,057 - ERROR - Failed to query channel 4 parameter 740: WriteFile failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
202

TPG366 disconnected.
